In [2]:
from src.utils.io import duckdb_s3

In [3]:
db = duckdb_s3(memory_limit='12GB', threads=4)

In [4]:
inpath = 's3://rcm-silver/sicor/'
labels = 's3://rcm-gold/labels/'
features = 's3://rcm-gold/features/'

In [ ]:
## Base safra

deltas = range(0, 25)
case_cols = ",\n         ".join(
    f'CASE WHEN first_default IS NOT NULL AND {d} >= first_default THEN 1 ELSE 0 END AS "{d}"'
    for d in deltas
)

sql = f"""
COPY (
  WITH op AS (
    SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, DT_EMISSAO
    FROM read_parquet('{inpath}operacao_basica.parquet')
  ),
  def_codes AS (
    SELECT "#CODIGO" AS cod
    FROM read_parquet('{inpath}situacao_operacao.parquet')
    WHERE DESCRICAO IN ('Inadimplente','Inscrita em Dívida Ativa da União','Baixada como Prejuízo')
  ),
  sld AS (
    SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem,
           ANO_BASE, MES_BASE, CD_SITUACAO_OPERACAO
    FROM read_parquet('{inpath}saldos.parquet')
  ),
  mut AS (
    SELECT "REF_BACEN" AS ref_bacen, CD_CPF_CNPJ AS mutuario
    FROM read_parquet('{inpath}mutuarios.parquet')
  ),
  base AS (
    SELECT s.ref_bacen, s.nu_ordem, m.mutuario,
           datediff('month', date_trunc('month', o.DT_EMISSAO),
                    make_date(TRY_CAST(s.ANO_BASE AS INT), TRY_CAST(s.MES_BASE AS INT), 1)) AS delta,
           CASE WHEN s.CD_SITUACAO_OPERACAO IN (SELECT cod FROM def_codes) THEN 1 ELSE 0 END AS situacao
    FROM sld s JOIN op o USING (ref_bacen, nu_ordem)
    JOIN mut m ON s.ref_bacen = m.ref_bacen
  ),
  fd AS (
    SELECT ref_bacen, nu_ordem, mutuario,
           MIN(CASE WHEN situacao = 1 THEN delta END) AS first_default
    FROM base
    WHERE delta BETWEEN 0 AND 24
    GROUP BY ref_bacen, nu_ordem, mutuario
  )
  SELECT ref_bacen, nu_ordem, mutuario,
         {case_cols}
  FROM fd
) TO '{labels}base_safra.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [40]:
db.execute(f"SELECT * FROM read_parquet('{outpath}labels/base_safra.parquet') LIMIT 5").df()

,ref_bacen,nu_ordem,mutuario,0,1,2,3,4,5,6,...,15,16,17,18,19,20,21,22,23,24
0,165204,3,42143969449,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,87625,3,06547923304,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,638022,1,59034319920,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,63308,1,06333413470,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,278895,1,88332675472,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
db.execute(f"SELECT MEAN(\"18\") FROM read_parquet('{labels}base_safra.parquet')").df()

,"mean(""18"")"
0,0.010727


In [ ]:
## Target no mês 18

sql = f"""
COPY (
    WITH op AS (
        SELECT \"#REF_BACEN\" AS ref_bacen, NU_ORDEM AS nu_ordem, DT_EMISSAO
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    def_codes AS (
        SELECT \"#CODIGO\" AS cod
        FROM read_parquet('{inpath}situacao_operacao.parquet')
        WHERE DESCRICAO IN ('Inadimplente','Inscrita em Dívida Ativa da União','Baixada como Prejuízo')
    ),
    sld AS (
        SELECT \"#REF_BACEN\" AS ref_bacen, NU_ORDEM AS nu_ordem,
            ANO_BASE, MES_BASE, CD_SITUACAO_OPERACAO
        FROM read_parquet('{inpath}saldos.parquet')
    ),
    mut AS ( \
        SELECT REF_BACEN as ref_bacen, CD_CPF_CNPJ AS mutuario
        FROM read_parquet('{inpath}mutuarios.parquet')
    ),
    base AS (
        SELECT s.ref_bacen, s.nu_ordem, m.mutuario,
            datediff('month', date_trunc('month', o.DT_EMISSAO),
                        make_date(TRY_CAST(s.ANO_BASE AS INT), TRY_CAST(s.MES_BASE AS INT), 1)) AS delta,
            CASE WHEN s.CD_SITUACAO_OPERACAO IN (SELECT cod FROM def_codes) THEN 1 ELSE 0 END AS situacao
        FROM sld s
        JOIN op o ON s.ref_bacen = o.ref_bacen AND s.nu_ordem = o.nu_ordem
        JOIN mut m ON s.ref_bacen = m.ref_bacen
    )
    SELECT ref_bacen, nu_ordem, mutuario,
            MAX(CASE WHEN delta BETWEEN 0 AND 18 THEN situacao ELSE 0 END) AS target_18m
    FROM base
    GROUP BY ref_bacen, nu_ordem, mutuario
) TO '{labels}target_18m.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
db.execute(f"SELECT * FROM read_parquet('{labels}target_18m.parquet') LIMIT 5").df()

,ref_bacen,nu_ordem,mutuario,target_18m
0,1005335,1,99904020353,0
1,174629,1,11689153504,0
2,201177,1,01717889310,0
3,328857,3,66171180491,0
4,295234,2,09663797304,0


In [ ]:
db.execute(f"SELECT MEAN(target_18m) FROM read_parquet('{labels}target_18m.parquet')").df()

,mean(target_18m)
0,0.010724


In [ ]:
## Feature Engineering
# Tipo do Produtor
# Identificador: Ref_Bacen e Mutuário 
# Descrição: Tipo de produtor rural, conforme definido pelo Ministério do Desenvolvimento Agrário e Agricultura Familiar (MDA). Definidas as categorias 'Produtor Rural PF/PJ' (cod 1), 'Extrativista' (cod 17), 
    # 'Pescador' (cod 12), 'Sílvicola/Indígena' (cod 11), 'Cooperativa de Produção Agropecuária' (cod 2 e 3 - Condição de Sociedade prestadora de serviço ou Produtor Rural), 'Aquicultor' (cod 13), 'Silvicultor' (cod 16),
    # 'Produtor PF/PJ de Mudas/Sementes/Embriões' (cod 6), 'Quilombola Rural' (cod 18), 'Agroindústria' (cod 9), 'Outros Produtores' (cod 4, 5, 7, 8, 10, 14, 15 - Inclui Associação de produtores rurais, 
    # Atividade de pesquisa agropecuária PF/PJ, Prestador de serviços agropecuários, Beneficiador, Cerealista, Torrefadora/Indústria e Exportador de café)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, mutuario, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    mut AS (
        SELECT REF_BACEN AS ref_bacen, CD_CPF_CNPJ AS mutuario, CD_TIPO_BENEFICIARIO AS tipo
        FROM read_parquet('{inpath}mutuarios.parquet')
    ),
    tipo AS (
        SELECT CAST("#CODIGO" AS INT) AS tipo, DESCRICAO AS tipo_produtor
        FROM read_parquet('{inpath}tipo_beneficiario.parquet')
    )
    SELECT safra.ref_bacen, safra.mutuario, safra.target_18m, 
        CASE 
            WHEN tipo.tipo_produtor IS NULL THEN NULL
            WHEN tipo.tipo_produtor = 'Produtor rural (pessoa física ou jurídica) - (MCR 1-4-1-a)' THEN 'Produtor Rural PF/PJ'
            WHEN tipo.tipo_produtor = 'Extrativista (MCR 2-1-15)' THEN 'Extrativista'
            WHEN tipo.tipo_produtor = 'Pescador (MCR 2-1-15 e 20)' THEN 'Pescador'
            WHEN tipo.tipo_produtor = 'Silvícola/indígena (MCR 1-4-3)' THEN 'Sílvicola/Indígena'
            WHEN tipo.tipo_produtor IN (
                'Cooperativa de produção agropecuária – na condição de sociedade prestadora de serviços de natureza agropecuária aos seus cooperados (MCR 5-1-2-b e c)',
                'Cooperativa de produção agropecuária – na condição de produtor rural - (MCR 5-1-2-a)'
            ) THEN 'Cooperativa de Produção Agropecuária'
            WHEN tipo.tipo_produtor = 'Aquicultor (MCR 2-1-20)' THEN 'Aquicultor'
            WHEN tipo.tipo_produtor = 'Silvicultor (MCR 10-2-2-a-III)' THEN 'Silvicultor'
            WHEN tipo.tipo_produtor = 'Pessoa física ou jurídica produtora de mudas, sementes, sêmen para inseminação artificial e embriões (MCR 1-4-2-a e b)' THEN 'Produtor PF/PJ de Mudas/Sementes/Embriões'
            WHEN tipo.tipo_produtor = 'Quilombola rural (MCR 10-2-2-b-II)' THEN 'Quilombola Rural'
            WHEN tipo.tipo_produtor = 'Agroindústria (MCR 1-4-2A-a)' THEN 'Agroindústria'
            ELSE 'Outros Produtores' 
        END AS tp_produtor
    FROM safra
    JOIN mut ON safra.ref_bacen = mut.ref_bacen AND safra.mutuario = mut.mutuario
    JOIN tipo ON mut.tipo = tipo.tipo
) TO '{features}tp_produtor.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
## Feature Engineering
# Categoria do Emitente
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: Definição do tamanho do produtor em relação ao valor contratado da operação e área produtiva. Segregado em 'Pequeno Produtor Rural' (cod 2222), 'Médio Produtor Rural' (cod 3333), 'Grande Produtor Rural' (cod 4444) 
    # e 'Outras Categorias' (cod 5555, 7777, 8888 - Categorias já desativadas)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_CATEG_EMITENTE
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    cat AS (
        SELECT "#CODIGO" AS cod, DESCRICAO AS categoria_emitente
        FROM read_parquet('{inpath}categoria_emitente.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
    CASE
        WHEN cat.categoria_emitente IS NULL THEN NULL
        WHEN cat.categoria_emitente IN ('Pequeno Produtor Rural','Médio Produtor Rural','Grande Produtor Rural') THEN cat.categoria_emitente
        ELSE 'Outras Categorias'
    END AS cat_emitente
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN cat ON op.CD_CATEG_EMITENTE = cat.cod
) TO '{features}cat_emitente.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
## Feature Engineering
# Instrumento de Crédito
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: Definição do instrumento de crédito utilizado para originar a operação. Segregado em 'Cédula Rural Pignoratícia' (cod 1), 'Cédula Rural Hipotecária' (cod 2), 'Cédula Rural Pignoratícia e Hipotecária' (cod 3),
    # 'Nota de Crédito Rural' (cod 4), 'Cédula de Crédito Bancário' (cod 5), 'Nota Promissória Rural' (cod 7), 'Contrato de Crédito Rural' (cod 10) e 'Demais Instrumentos de Crédito' (cod 6, 8, 9, 99 - Inclui Duplicata Rural,
    # Outros Instrumentos de Crédito Rural, Cédula de Produto Rural-CPR (AVAL) - ENCERRADO e Termo de Adesão ao Proagro)

sql = f"""
COPY(
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_INST_CREDITO
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    cred AS (
        SELECT CAST("#CODIGO" AS INT) AS cod, DESCRICAO AS inst_credito
        FROM read_parquet('{inpath}instrumento_credito.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
    CASE
        WHEN cred.inst_credito IS NULL THEN NULL
        WHEN cred.inst_credito IN (
            'Cédula Rural Pignoratícia','Cédula Rural Hipotecária','Cédula Rural Pignoratícia e Hipotecária','Nota de Crédito Rural','Cédula de Crédito Bancário','Nota Promissória Rural','Contrato de Crédito Rural'
        ) THEN cred.inst_credito
        ELSE 'Demais Instrumentos de Crédito'
    END AS instrum_credito
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN cred ON op.CD_INST_CREDITO = cred.cod
) TO '{features}instrum_credito.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [83]:
## Feature Engineering
# Fonte de Recursos
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: Definição da fonte de recursos utilizada para originar a operação. Segregado em 'Recursos Controlados' (cod 100, 201, 202, 205, 222, 225, 226, 260, 300, 301, 302, 304, 403, 431, 440, 450, 501, 502, 503, 505, 507, 
    # 520, 600, 650, 680, 800, 910) e 'Recursos Livres' (cod 303, 402, 405, 430, 451, 506, 850, 900, 990 - Inclui POUPANÇA RURAL - LIVRE, RECURSOS LIVRES, FUNDO DE COMMODITIES, CAPTAÇÃO EXTERNA, 
    # LETRA DE CRÉDITO DO AGRONEGÓCIO - TAXA LIVRE, BNDES LIVRE, INSTR HIBRIDO CAPITAL DÍVIDA - LIVRE, ATIVIDADE NÃO FINANCIADA ENQUADRADA NO PROAGRO e OUTRAS FONTES DE RECURSOS NÃO ESPECIFICADAS)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_FONTE_RECURSO
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    cred AS (
        SELECT CAST("#CODIGO" AS INT) AS cod, DESCRICAO AS fonte_recurso
        FROM read_parquet('{inpath}fonte_recursos.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
    CASE
        WHEN cred.fonte_recurso IS NULL THEN NULL
        WHEN cred.fonte_recurso IN (
            'RECURSOS LIVRES EQUALIZÁVEIS','FACULDADE DE APLICAÇÃO - COMPULSÓRIO','TESOURO NACIONAL','OBRIGATÓRIOS - MCR 6.2','POUPANÇA RURAL - CONTROLADOS - SUBVENÇÃO ECONÔMICA','POUPANÇA RURAL - CONTROLADOS - CONDIÇÕES MCR 6.2',
            'POUPANÇA RURAL - CONTROLADOS - FATOR DE PONDERAÇÃO','FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORTE (FNO)','FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORDESTE (FNE)','FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO CENTRO-OESTE (FCO)',
            'BNDES/FINAME - EQUALIZÁVEL','FAT - FUNDO DE AMPARO AO TRABALHADOR','FUNCAFE-FUNDO DE DEFESA DA ECONOMIA CAFEEIRA','FUNDO DE TERRAS E DA REFORMA AGRÁRIA','INCRA','GOVERNOS E FUNDOS ESTADUAIS OU MUNICIPAIS','PIS/PASEP',
            'INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - EQUALIZÁVEL','COMPULSÓRIO SOBRE RECURSOS À VISTA - REFORÇO DO INVESTIMENTO (CIRC 3.745)','LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - TAXA FAVORECIDA',
            'Exigibilidade Adicional dos Recursos à Vista - Resolução 5030 ENCERRADO','LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - CONTROLADOS - SUBVENÇÃO ECONÔMICA - MCR 6-7-7A-"b"-I','Exigibilidade Adicional dos Recursos à Vista - Resolução 5087 - ENCERRADO',
            'Exigibilidade Adicional dos Recursos à Vista - Resolução 5157','Fundo Nacional sobre Mudança do Clima (Fundo Clima) - RES CMN 5.130/2024','EXIGIBILIDADE ADICIONAL DOS RECURSOS À VISTA','EXIGIBILIDADE ADICIONAL DA POUPANÇA RURAL'
        ) THEN 'Recursos Controlados'
        ELSE 'Recursos Livres'
    END AS fonte_recurso
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN cred ON op.CD_FONTE_RECURSO = cred.cod
) TO '{features}fonte_recurso.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [89]:
## Feature Engineering
# Estado da Operação
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: UF's de originação da operação, conforme definido pelo Banco Central do Brasil. Segregado nas 27 UF's do Brasil

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_ESTADO as uf
        FROM read_parquet('{inpath}operacao_basica.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, op.uf
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
) TO '{features}uf_operacao.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [7]:
## Feature Engineering
# Seguro Rural
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: Definição do tipo de seguro rural utilizado na operação. Segregado em 'Proagro' (cod 1 e 2), 'Sem seguro' (cod 0 e 9) e 'Outros seguros' (cod 3)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_TIPO_SEGURO
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    seg AS (
        SELECT CAST("#CODIGO" AS INT) AS cod, DESCRICAO AS seguro
        FROM read_parquet('{inpath}tipo_seguro.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, 
        CASE
            WHEN seg.seguro IS NULL THEN NULL
            WHEN seg.seguro IN ('Não se aplica', 'Sem adesão a seguro') THEN 'Sem seguro'
            WHEN seg.seguro IN ('Proagro mais', 'Proagro tradicional') THEN 'Proagro'
            ELSE 'Outros seguros'
        END AS tipo_seguro
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN seg ON op.CD_TIPO_SEGURO = seg.cod
) TO '{features}tipo_seguro.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [23]:
## Feature Engineering
# Finalidade
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: Definição da finalidade do recurso contratado na operação. Segregado em 'Custeio', 'Investimento', 'Comercialização', 'Industrialização' e 'Outras Finalidades' (outras finalidades utilizado para incluir renegociações)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_EMPREENDIMENTO as empreendimento
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ), 
    emp AS (
        SELECT "#CODIGO" AS cod, FINALIDADE
        FROM read_parquet('{inpath}empreendimento.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
        CASE WHEN emp.FINALIDADE IN ('Custeio','Investimento','Comercialização','Industrialização') THEN emp.FINALIDADE 
        ELSE 'Outras finalidades' END AS finalidade
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN emp ON op.empreendimento = emp.cod
) TO '{features}finalidade.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [24]:
## Feature Engineering
# Atividade
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: Definição da atividade da operação. Segregado em 'Agrícola', 'Pecuário(a)' e 'Outras Atividades' (outras atividades utilizado para incluir renegociações)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_EMPREENDIMENTO as empreendimento
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ), 
    emp AS (
        SELECT "#CODIGO" AS cod, ATIVIDADE
        FROM read_parquet('{inpath}empreendimento.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
        CASE WHEN emp.ATIVIDADE IN ('Agrícola','Pecuário(a)') THEN emp.ATIVIDADE 
        ELSE 'Outras atividades' END AS atividade
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN emp ON op.empreendimento = emp.cod
) TO '{features}atividade.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
## Feature Engineering
# Modalidade
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição: 'Outras modalidades' (Inclui renegociações, algumas modalidades encerradas e modalidades de tarifas e custos)

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_EMPREENDIMENTO as empreendimento
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ), 
    emp AS (
        SELECT "#CODIGO" AS cod, FINALIDADE, MODALIDADE
        FROM read_parquet('{inpath}empreendimento.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
        CASE
            WHEN emp.MODALIDADE IS NULL THEN NULL
            WHEN emp.MODALIDADE IN (
                'APICULTURA','AQUISIÇÃO E MANUTENÇÃO DE ANIMAIS','AVICULTURA','BUBALINOCULTURA','CAPRINOCULTURA','CRIA DE ANIMAIS','CUNICULTURA E DEMAIS ROEDORES','EQUINOCULTURA','EXPLORAÇÃO SOB REGIME DE INTEGRAÇÃO - ENCERRADO',
                'LAVOURA','MANUTENÇÃO/CRIAÇÃO DE ANIMAIS (CRIA)','MANUTENÇÃO/CRIAÇÃO DE ANIMAIS (RECRIA E ENGORDA)','MINHOCULTURA','OVINOCULTURA','SERICICULTURA','SUINOCULTURA','Agroindústria Familiar (MCR 10-11)',
            ) OR (emp.MODALIDADE = 'AQUICULTURA' AND emp.FINALIDADE = 'Custeio') 
            OR (emp.MODALIDADE = 'BOVINOCULTURA' AND emp.FINALIDADE = 'Custeio') 
            OR (emp.MODALIDADE = 'EXTRATIVISMO DE ESPÉCIES NATIVAS' AND emp.FINALIDADE = 'Custeio')
            OR (emp.MODALIDADE = 'FLORESTAMENTO E REFLORESTAMENTO' AND emp.FINALIDADE = 'Custeio')
            OR (emp.MODALIDADE = 'PASTAGEM' AND emp.FINALIDADE = 'Custeio')
            OR (emp.MODALIDADE = 'AQUISIÇÃO DE INSUMOS PARA INDÚSTRIA FAMILIAR' AND emp.FINALIDADE = 'Custeio')
            OR (emp.MODALIDADE = 'BENEFICIAMENTO OU INDUSTRIALIZAÇÃO' AND emp.FINALIDADE = 'Custeio')
            OR (emp.MODALIDADE = 'PESCA' AND emp.FINALIDADE = 'Custeio')
            OR (emp.MODALIDADE = 'SERVIÇOS PROFISSIONAIS/TÉCNICOS' AND emp.FINALIDADE = 'Custeio')
            THEN 'Manutenção de ciclo'
            WHEN emp.MODALIDADE IN (
                'AQUISIÇÃO DE ATIVOS OPERACIONAIS','AQUISIÇÃO DE PROPRIEDADES RURAIS','AQUISIÇÃO DE VEÍCULOS','Financiamento para Aquisição da Produção/Materia P','MELHORAMENTO DAS EXPLORAÇÕES',
                'MÁQUINAS, EQUIPAMENTOS, MATERIAIS E UTENSÍLIOS',
            ) OR (emp.MODALIDADE = 'AQUICULTURA' AND emp.FINALIDADE = 'Investimento')
            OR (emp.MODALIDADE = 'EXTRATIVISMO DE ESPÉCIES NATIVAS' AND emp.FINALIDADE = 'Investimento')
            OR (emp.MODALIDADE = 'AQUISIÇÃO DE INSUMOS PARA INDÚSTRIA FAMILIAR' AND emp.FINALIDADE = 'Investimento')
            OR (emp.MODALIDADE = 'BENEFICIAMENTO OU INDUSTRIALIZAÇÃO' AND emp.FINALIDADE = 'Investimento')
            OR (emp.MODALIDADE = 'PESCA' AND emp.FINALIDADE = 'Investimento')
            OR (emp.MODALIDADE = 'SERVIÇOS PROFISSIONAIS/TÉCNICOS' AND emp.FINALIDADE = 'Investimento')
            THEN 'Investimento em máquinas/infraestrutura/ativos'
            WHEN emp.MODALIDADE IN (
                'AQUISIÇÃO DE ANIMAIS','AQUISIÇÃO DE ANIMAIS DE SERVIÇO','AQUISIÇÃO DE ANIMAIS DE SERVIÇO (USO AGRICULTURA)','FORMAÇÃO DE CULTURAS PERENES','IMPLANTAÇÃO E MELHORAMENTO'
            ) OR (emp.MODALIDADE = 'BOVINOCULTURA' AND emp.FINALIDADE = 'Investimento')
             OR (emp.MODALIDADE = 'PASTAGEM' AND emp.FINALIDADE = 'Investimento')
            THEN 'Investimento em material biológico'
            WHEN emp.MODALIDADE IN (
                'Aquisição de Matéria Prima direto do Produtor/Coop','CPR (CÉDULA DE PRODUTO RURAL)','DESCONTO (NPR E DR)','ESTOCAGEM','FAC - Financiamento para Aquisição de Café','FEE (EX-LEC)','FEPM (EX-EGF) - encerrado',
                'FGPP-Financiamento para Garantia de Preços ao Prod','FINANCIAMENTO PARA PROTEÇÃO DE PREÇOS EM OPERAÇÕES','PRÉ-COMERCIALIZAÇÃO - encerrado'
            ) 
            OR (emp.MODALIDADE = 'AQUICULTURA' AND emp.FINALIDADE = 'Comercialização')
            OR (emp.MODALIDADE = 'PESCA' AND emp.FINALIDADE = 'Comercialização')
            OR (emp.MODALIDADE = 'SERVIÇOS PROFISSIONAIS/TÉCNICOS' AND emp.FINALIDADE = 'Comercialização')
            THEN 'Estocagem/processamento/proteção de preço'
            WHEN emp.MODALIDADE IN (
                'ATENDIMENTO A COOPERADOS','COOPERATIVAS DE CRÉDITO (SINGULAR OU CENTRAL) - en','Crédito para Cooperativa Pronaf MCR 10-3-1A','FINANCIAMENTO PROCAP-AGRO','INTEGRALIZAÇÃO DE COTAS PARTES'
            ) THEN 'Cooperativismo/linhas de apoio'
            ELSE 'Outras modalidades'
        END AS modalidade
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN emp ON op.empreendimento = emp.cod
) TO '{features}modalidade.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [12]:
## Feature Engineering
# Produto
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_EMPREENDIMENTO as empreendimento
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ), 
    emp AS (
        SELECT "#CODIGO" AS cod, MODALIDADE, FINALIDADE, PRODUTO, VARIEDADE
        FROM read_parquet('{inpath}empreendimento.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
        CASE
            WHEN emp.PRODUTO IS NULL THEN NULL
            WHEN emp.PRODUTO IN ('SOJA','ARROZ','FEIJÃO','AGROARTESANATO','AGROINDÚSTRIA') THEN emp.PRODUTO
            WHEN emp.PRODUTO IN ('MILHO','MILHO SILAGEM') THEN 'MILHO'
            WHEN emp.PRODUTO IN ('TRIGO','TRIGO SILAGEM','TRIGO SARRACENO/MOURISCO') THEN 'TRIGO'
            WHEN emp.PRODUTO IN ('ALGODÃO','CAROÇO DE ALGODÃO') THEN 'ALGODÃO'
            WHEN emp.PRODUTO IN ('CAFÉ','RECUPERAÇÃO DE CAFEZAIS') THEN 'CAFÉ'
            WHEN emp.PRODUTO IN ('BOVINOS','Confinamento de bovinos "free stall','SEBO BOVINO','Aquisição de Sistemas para Rastreabilidade de bovinos e bubalinos') THEN 'BOVINOCULTURA'
            WHEN emp.PRODUTO IN ('GRANJAS DE SUÍNOS','SUINOCULTURA','SUÍNOS') THEN 'SUÍNOCULTURA'
            WHEN emp.PRODUTO IN (
                'ABACATE','ABACAXI','ABRICÓ (DAMASCO)','ACEROLA','AMEIXA','AMORA','ARAÇA','ATEMOIA','AÇAÍ','BABAÇU','BANANA','BURITI','CACAU','CAMU-CAMU','CAJU','CAJÁ','CAMAPU (PHYSALIS)','CAQUI','CARAMBOLA','CEREJA','CHERIMOIA',
                'CIDRA','COCO','COCO-DA-BAIA','CUPUAÇU','FIGO','FRAMBOESA','FRUTAS DIVERSAS N. E.','GABIROBA','GOIABA','GRAVIOLA','GROSELHA','GUARANÁ','JABUTICABA','JACA','JILÓ','LARANJA','LIMA','LIMÃO','LONGAN','MAMÃO','MANGA',
                'MANGABA','MANGOSTÃO','MARACUJÁ','MARMELO','MAÇÃ','MELANCIA','MELÃO','MIRTILO','MORANGO','MURICI','NECTARINA','NESPERA','PEQUI','PERA','PINHA (ATA, FRUTA-DO-CONDE, ANONA)','PITANGA','PITAYA','POMELO','PÊSSEGO',
                'QUIUÍ (KIWI)','ROMÃ','SAPOTI','TAMARINDO','TANGERINA','TAPEREBÁ','TORANJA','TUCUM','UMBU','UVA','CÂMARA FRIA PARA ARMAZENAMENTO DE FRUTAS'
            ) THEN 'FRUTICULTURA'
            WHEN emp.PRODUTO IN (
                'ABOBRINHA','ABÓBORA-MORANGA','ACELGA','AGRIÃO','AIPO','ALCACHOFRA','ALFACE','ALFAFA','ALHO','ALHO PORÓ','ALMEIRÃO','ASPARGO','BATATA','BATATA ASTERIX','BATATA-DOCE','BATATA-INGLESA','BERINJELA',
                'BETERRABA','BRÓCOLOS (BRÓCOLIS)','CARÁ','CEBOLA','CEBOLINHA','CEBOLINHA VERDE','CENOURA','CHICORIA','CHUCHU','COENTRO','COUVE','COUVE-FLOR','ESCAROLA','ESPINAFRE','HORTALIÇAS','Hortaliça ora-pro-nóbis','INHAME',
                'MANDIOCA (AIPIM, MACAXEIRA)','MANDIOQUINHA (BATATA: BAROA, SALSA, AIPO)','MAXIXE','MOSTARDA','NABO','OLERÍCOLAS','PEPINO','PIMENTÃO','QUIABO','RABANETE','REPOLHO','RÚCULA','SALSA','TAIOBA','TOMATE','VAGEM'
            ) THEN 'OLERICULTURA/HORTALIÇAS'
            WHEN emp.PRODUTO IN (
                'ALGICULTURA (Cultivo de Algas)','Alevinos','Algas','Alimentador de Peixe','CAMARÃO','CAMARÃO E/OU LAGOSTA','CARCINICULTURA','CARCINICULTURA (Cultivo de Camarão e Lagosta)','CRUSTÁCEOS','MEXILHÃO','MARISCOS',
                'Camarão e Lagosta','Descamadora de Peixe','Descascador de Camarão e Lagosta','Girinos','MALACOCULTURA (Cultivo e Criação de Moluscos)','MARICULTURA (Cultivo de Marisco)','Mariscos','PESCADO IN NATURA',
                'MITILICULTURA (Cultivo de Mexilhão)','MOLUSCOS (Caramujos, Lulas etc)','Mexilhões','OSTRAS','OSTREICULTURA','OSTREICULTURA (Cultivo de Ostras)','PEIXE','PESCA','PESCADO','PISCICULTURA','RÃ','RANICULTURA',
                'PESCADO (ARMAZENAMENTO, ACONDICIONAMENTO E PRESERVACAO, INCLUSIVE SEGURO, IMPOSTOS, FRETES ETC)','PISCICULTURA (CULTIVO DE PEIXES)','PISCICULTURA (Cultivo de Peixe)','RANICULTURA (Cultivo de Rã)',
                'Petrechos para Pesca (anzóis, iscas, cordas, bóias, combustivel, redes, mão-de-obra etc)','Aerador','Armação para Barco de Pesca','CLASSIFICADOR DE PESCADO','COZEDOR DE PESCADO','DEPÓSITO PARA RAÇÕES',
                'DESPOLPADOR DE PESCADO','Embarcação Grande (a partir de 100 A/B)','Embarcação Média (acima de 20 e abaixo de 100 A/B)','Embarcação Pequena (até 20 A/B)','Esteira','Estufa','Evisceradora',
                'MESA PARA DESCABEÇAMENTO DE PESCADO','MESA PARA RETIRADA DE PELE, ESCAMA E CARCAÇA DE PESCADO','Mesa para Filetagem','SEPARADOR DE RESÍDUOS','Tanques Escavados','Tanques Redes',
                'PRODUTOS AQUICOLAS (Armazenamento, Acondicionamento e Preservação, inclusive Seguro, Impostos etc)','Unidade de Beneficiamento ou Processamento'
            ) THEN 'AQUICULTURA/PESCADO'
            WHEN emp.PRODUTO IN (
                'ABELHA','APICULTURA','ANIMAIS SILVESTRES','ASININOS','AVESTRUZ','BICHO-DA-SEDA','BÚFALOS (BUBALINOS)','CANINOS','CAPRINOS','CHINCHILAS','COELHO','CUNICULTURA','EQUINOS','LÃ','MEL','MINHOCA','MELIPONICULTURA',
                'Caixas de abelhas, favos, centrifugas p/ extração de mel, fumegadores','Instalações para Aves, Suínos e Coelhos','MUARES','OUTROS ANIMAIS','OVINOS','SERICICULTURA','SIRGARIAS','TRIPAS','VISCACHAS','COURO','LEITE',
                'Balança para Animais','Caminhões Frigoríficos','INSEMINAÇÃO ARTIFICIAL','Matrizes e Reprodutores','Medicamentos, Rações e Insumos','RESÍDUOS DE PRODUÇÃO ANIMAL','Raspador','SAL',
                'VACINAS, SAIS MINERAIS E MEDICAMENTOS'
            ) THEN 'OUTRAS PECUÁRIAS'
            WHEN emp.PRODUTO IN (
                'ACÁCIA NEGRA','AGAVE (SISAL)','ANDIROBA','ARAUCÁRIA','BUCHA VEGETAL','Bambu','BARU','CASTANHA DE BARU','CASTANHA DE CAJU','CASTANHA-DO-BRASIL','CAMBARÁ','CARNAÚBA','CEDRINHO','CEDRO','CURAUÁ','DENDÊ',
                'Cumaru/Champanhe','EUCALIPTO','GARAPEIRA','GUARIROBA','JATOBÁ','JUTA','Kiri (Paulownia spp)','LINHO','MADEIRA','MURUMURU','Macaúba','NOZ','OLIVA (AZEITONA)','PALMITO (PUPUNHA,AÇAI)','PARICÁ','PIAÇABA (PIAÇAVA)',
                'PINHÃO','PINUS','PORONGO (CUIA,CABAÇA)','PRACAXI','PUPUNHA','RAMI','SERINGUEIRA','TUNGUE','VIME','ÓLEO VEGETAL','TANINO','EQUIPAMENTOS E UTENSÍLIOS PARA EXTRATIVISMO DE ESPÉCIES NATIVAS'
            ) OR (emp.PRODUTO = 'FLORESTAMENTO E REFLORESTAMENTO' AND emp.VARIEDADE IN (
                'EUCALIPTO DUNNII','EUCALIPTO GRANDIS','EUCALIPTO GLOBULUS','EUCALIPTO SALIGNA','EUCALIPTO VIMINALIS','EUCALIPTO BENTHAMII'
            )) THEN 'SILVICULTURA/EXTRATIVISMO'
            WHEN emp.PRODUTO IN (
                'AMENDOIM','AVEIA','CANOLA','CARINATA','CENTEIO','CEVADA','CHIA','COLZA','ERVILHA','GERGELIM','GIRASSOL','GRÃO-DE-BICO','GRÃOS','LENTILHA','LICHIA (LECHIA)','LINHAÇA','LÚPULO','PAINÇO','QUINOA'
            ) THEN 'OUTROS GRÃOS/OLEAGINOSAS'
            WHEN emp.PRODUTO IN (
                'AROEIRA (PIMENTA-ROSA)','AÇAFRÃO','CHÁ','COPAÍBA','CRAVO','CRAVO-DA-ÍNDIA','ERVA CIDREIRA (MELISSA)','ERVA-DOCE','ERVA-MATE','Ervas medicinais, aromáticas ou condimentares','GENGIBRE','HORTELÃ',
                'MALVA','MANJERICÃO','MENTA','MORINGA','NEEM','PATAUÁ','PIMENTA','PIMENTA-DO-REINO','PIXURIM','POEJO','SERRALHA','URUCUM'
            ) THEN 'CONDIMENTOS/ERVAS MEDICINAIS'
            WHEN emp.PRODUTO IN ('AVES','AVES EXCETO GALINÁCEOS','AVICULTURA','FRANGO','GALINÁCEOS','GRANJAS AVÍCOLAS','OVOS') THEN 'AVICULTURA'
            WHEN emp.PRODUTO IN (
                'AZEVEM','CAPIM','CROTALÁRIA','ERVILHACA - RALIÇA, VICIA SATIVA','ESTILOSANTES','FORRAÇÃO DE JARDIM','MAMONA','MILHETO','NIGER','PALMA','PASTAGEM','SORGO','TIFTON','TRITICALE'
            ) THEN 'PASTAGENS/FORRAGENS'
            WHEN emp.PRODUTO IN (
                'CITRONELA (CYMBOPOGON NARDUS)','COGUMELO','ESTÉVIA','FLORESTAMENTO - TRATOS CULTURAIS','FLORESTAMENTO E REFLORESTAMENTO','FUMO','MUDAS DIVERSAS','OUTRAS CULTURAS','OUTRAS LAVOURAS','PLANTAS PARA INFUSÃO',
                'ADUBAÇÃO ORGÂNICA/MINERAL, CALAGEM, SUBSTRATOS INERTES(PEDRA, AREIA, VERMICULITA, SILTE, ARGILA ETC)','AQUISIÇÃO DE FERRAMENTA PORTÁTIL MANUAL PARA TRATOS CULTURAIS','CONSERVAÇÃO DE SOLOS',
                'CERCAS, ARAMADOS, TELHAS, TELAS PARA SOMBREAMENTO E COBERTURAS DE SOLO','COBERTURAS DE SOLO (PLÁSTICAS, TNT, TECIDOS, SERRAGEM, PALHADAS DE CAPIM E DE GRÃOS ETC)','DESPOLPADOR','CANA-DE-AÇÚCAR','AÇUCAR'
                'EQUIPAMENTOS E UTENSILIOS PARA AGRICULTURA DE PRECISÃO','ESTUFAS/VIVEIROS (ILUMIN. ARTIFICIAL, MUDAS, SEMENTES, SACOS, TALAGARÇAS, BANDEJAS, VASOS)','HIDROPONIA/FAZENDA VERTICAL (ALVENARIA, MADEIRA, AÇO, ETC)',
                'IRRIGAÇÃO/LIXIVIAÇÃO (GOTEJADOR, ASPERSOR, NEBULIZADOR, EXAUSTOR, VENTILADOR, MANGUEIRAS, CANAIS ET)','SECADOR','TENDA, GALPÃO, TÚNEL PLÁSTICO (ABRANGE LONAS, FILMES, LONGARINAS, ESTACAS E MAT. SUSTENTAÇÃO)',
                'TULHA'
            ) OR (emp.PRODUTO = 'AQUISIÇÃO DE INSUMOS' AND emp.MODALIDADE = 'LAVOURA') THEN 'OUTRAS CULTURAS'
            WHEN emp.PRODUTO IN ('CRISÂNTEMO','FLORES','GRAMA','PALMEIRA','PALMÁCEA','PLANTAS ORNAMENTAIS','SANSÃO-DO-CAMPO') THEN 'ORNAMENTAIS'
            ELSE 'OUTROS PRODUTOS'
        END AS produto
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN emp ON op.empreendimento = emp.cod
) TO '{features}produto.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
## OUTROS PRODUTOS
'ABERTURA DE GLEBAS RURAIS','ADIANTAMENTOS A COOPERADOS POR CONTA DE PRODUTOS ENTREGUES PARA VENDA','ADUBAÇÃO INTENSIVA DO SOLO','ANTECIPAÇÃO DE RECURSOS ORIGINÁRIOS DE TAXA DE RETENÇÃO',
'AQUISIÇÃO DE AQUECEDORES, GERADORES, INCINERADORES, COMPRESSORES, VENTILADORES E APAR. DE AR CONDIC.','AQUISIÇÃO DE BENS PARA FORNECIMENTO AOS COOPERADOS',
'AQUISIÇÃO DE BENS PARA PRESTAÇÃO DE SERVIÇOS EXCLUSIVAMENTE EM EXPLORAÇÕES RURAIS','AQUISIÇÃO DE DESTILADORES, FILTROS, DEPURADORES E DOSADORES','AQUISIÇÃO DE EMPILHADEIRA(S)/TOMBADORA(S)',
'AQUISIÇÃO DE EQUIPAMENTO(S) TOPOGRAFICO(S)','AQUISIÇÃO DE EQUIPAMENTOS DE INFORMÁTICA E TELECOMUNICAÇÕES, INCLUINDO AQUISIÇÃO DE SOFTWARE','AQUISIÇÃO DE EQUIPAMENTOS PARA MANUTENÇÃO DE VEÍCULOS',
'AQUISIÇÃO DE EQUIPAMENTOS PARA MOLDAGEM, TORNOS, MOINHOS E PRENSA','AQUISIÇÃO DE FERRAMENTAS E TUBOS','AQUISIÇÃO DE INSUMOS','AQUISIÇÃO DE INSUMOS PARA FORNECIMENTO AOS COOPERADOS',
'AQUISIÇÃO DE MAQUINA(S)/EQUIPAMENTO(S) PARA EXTRAÇÃO/FABRICAÇÃO (INCLUINDO GUINCHOS E GUINDASTES)','AQUISIÇÃO DE MATERIAL DE ESTOCAGEM/TRANSPORTE/SACARIA/CAIXAS','AQUISIÇÃO DE MOTORES E ELEVADORES',
'AQUISIÇÃO DE UTENSÍLIOS DOMÉSTICOS PARA USO EM MEIO RURAL','ARMAZÉM','ARMAZÉM, DEPÓSITO, SILO, GALPÃO, PAIOL, ESTUFA E INSTALAÇÕES CONGÊNERES','ATENDIMENTO DE DESPESAS','ATIVIDADE RURAL GERADORA DE RENDA PARA UNIDADE FAMILIAR',
'AVIAÇÃO AGRÍCOLA','AVIÕES','Aquecedor de Água (Boiler)','Ações de Sutentabilidade Ambiental e Energia Renovável','BASE PARA BALANÇA','BICICLETAS',
'BIODIGESTOR, ESTERQUEIRA, TANQUES DE OXIDAÇÃO BIOLÓGICA E TRATAMENTO DE ÁGUA E ESGOTO','BIOINSUMO','CAMINHÕES','CAMIONETAS','CAPITAL DE GIRO','CARRETAS, CARROÇAS E VAGÕES DE CARGA, FORRAGEIROS E DEMAIS','CERTIFICAÇÃO FLORESTAL',
'COLHEITA, DESTOCA E LIMPEZA DE FLORESTA PLANTADA','COLHEITADEIRAS, COLHEDEIRAS E ARRANCADEIRAS','CONSTRUÇÃO E AMPLIAÇÃO DE BENFEITORIAS E INSTALAÇÕES PERMANENTES','CORREÇÃO INTENSIVA DO SOLO','CORREÇÃO NÃO INTENSIVA','CULTIVADOR',
'CULTIVADORES MOTORIZADOS','CUSTEIO DE MODALIDADE NÃO AGROPECUÁRIA','Construção/Recuperação Barragem/Tanque, sistemas captação de água','DEPÓSITO E INSTALAÇÕES CONGÊNERES','DESMATAMENTO','DESTOCA','ELETRIFICAÇÃO RURAL',
'EMBARCAÇÕES','ERRADICAÇÃO DE CAFEZAIS','ESCOLAS RURAIS','FUNDIÁRIOS','INSTALAÇÃO/MONTAGEM/TRANSPORTE DE EQUIPAMENTO(S)','INSTALAÇÕES PARA INDUSTRIALIZAÇÃO E BENEFICIAMENTO','INTEGRALIZAÇÃO DE COTAS PARTES',
'INTEGRAÇÃO LAVOURA-PECUÁRIA-FLORESTA','INVESTIMENTO FIXO','IRRIGAÇÃO','Implantação de tecnologias de energia renovável, ambiental e pequenas aplicações hidroenergéticas','Indicador de renegociação','JIPES, FURGÕES E SEMELHANTES',
'LAGO ARTIFICIAL, TANQUE, BARREIROS, CANAIS, RESERV.ÁGUA POTAVEL','LAVADOR','MANUTENÇÃO DO PRODUTOR E SUA FAMÍLIA','MARGENS DE GARANTIA E AJUSTES DIÁRIOS EM VENDAS FUTURAS DE PRODUTOS AGROPECUÁRIOS','MATERIAIS INDUSTRIALIZADOS',
'MOTOCICLETAS E MOTONETAS','MÁQUINAS E IMPLEMENTOS','MÁQUINAS, APARELHOS E INSTRUMENTOS','MÁQUINAS, APARELHOS E INSTRUMENTOS PARA PROCESSAMENTO E BENEFICIAMENTO  DE  PRODUTOS','NÃO SE APLICA','OUTRAS APLICAÇÕES',
'OUTRAS APLICAÇÕES DE CUSTEIO AGRÍCOLA','OUTRAS APLICAÇÕS DE INVESTIMENTO AGRÍCOLA','OUTRAS MÁQUINAS','OUTROS MELHORAMENTOS','OUTROS PRODUTOS','OUTROS PRODUTOS BENEFICIADOS OU INDUSTRIALIZADOS NÃO ESPECIFICADOS','OUTROS VEÍCULOS',
'PAGAMENTO DE TAXAS E EMOLUMENTOS DAS BOLSAS DE MERCADORIAS E FUTURO','PAGTO DOS PRÊMIOS EM CONTRATOS DE OPÇÃO DE VENDA DE PRODUTOS AGROP EM BOLSAS DE MERC E FUTURO','PRESTAÇÃO DE SERVIÇOS NO MEIO RURAL','PROTEÇÃO DO SOLO',
'Perfuração Poço, Cacimba/Cisterna','Prestação de Assesoria Técnica e Empresarial / Consultoria e Elaboração de projetos e treinamentos','Prestação de Serviços Mecanizados MCR 4-4','QUINTAIS PRODUTIVOS PARA MULHERES RURAIS',
'REBOQUES, SEMIREBOQUES, CAÇAMBAS E CABINES','RECUPERAÇÃO AMBIENTAL','REFORMAS  DE  MÁQUINAS, APARELHOS, EQUIPAMENTOS  E VEÍCULOS, COMPRA DE PECAS E ACESSRIOS','REGULARIZAÇÃO FUNDIÁRIA DO IMÓVEL RURAL','RENEGOCIAÇÃO PARCIAL',
'RENEGOCIAÇÃO TOTAL','REPASSE INTERFINANCEIRO','Residências Rurais','SACARIA E/OU MATERIAL DE ACONDICIONAMENTO','SANEAMENTO FINANCEIRO','SILO','SISTEMAS DE CAPTAÇÃO, RETENÇÃO E APROVEITAMENTO DE ÁGUA',
'SISTEMAS DE PRODUÇÃO ORGÂNICA','SUPRIMENTO DE RECURSOS PARA ATENDIMENTO A COOPERADOS','TANQUE DE ARMAZENAMENTO','TECNOLOGIAS, EQUIPAMENTOS E SOFTWARES','TERREIROS','TRATOR','TRICICLOS E QUADRICLOS','TURISMO E LAZER RURAL',
'Teleférico Rural','Terraços, Porteiras, Mata-burros, Currais, Cochos, Cercas','UTILITÁRIOS','VEÍCULO AÉREO NÃO TRIPULADO (DRONE)','ÔNIBUS, MICRO-ÔNIBUS E VANS'

In [18]:
## Feature Engineering
# Cesta de Cultivo
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_EMPREENDIMENTO as empreendimento
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    emp AS (
        SELECT "#CODIGO" AS cod, CESTA
        FROM read_parquet('{inpath}empreendimento.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, emp.CESTA AS cesta_cultivo
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN emp ON op.empreendimento = emp.cod
) TO '{features}cesta_cultivo.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [33]:
## Feature Engineering
# Programa
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_PROGRAMA as programa
        FROM read_parquet('{inpath}operacao_basica.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
        CASE
            WHEN op.programa IS NULL THEN NULL
            WHEN op.programa = '0001' THEN 'PRONAF'
            WHEN op.programa = '0050' THEN 'PRONAMP'
            -- WHEN op.programa IN ('0777','0721','0722','0779','0730','0783','0776','0784','0735','0785','0786','0790') THEN 'RENEGOCIAÇÃO'
            ELSE 'DEMAIS PROGRAMAS'
        END AS programa_status
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
) TO '{features}programa.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [44]:
## Feature Engineering
# Tipo Irrigação
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_TIPO_IRRIGACAO as tipo_irrigacao
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    irrig AS (
        SELECT "#CODIGO" AS cod, DESCRICAO AS tipo_irrigacao
        FROM read_parquet('{inpath}tipo_irrigacao.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m,
        CASE
            WHEN irrig.tipo_irrigacao IS NULL THEN NULL
            WHEN irrig.tipo_irrigacao NOT IN ('Não Irrigado','Não se aplica') THEN irrig.tipo_irrigacao
            WHEN irrig.tipo_irrigacao IN ('Irrigação com cobertura contra a seca MCR 12-2-3-"c') THEN 'Cobertura contra a seca'
            ELSE 'Não irrigado/Não se aplica'
        END AS tipo_irrigacao
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN irrig ON op.tipo_irrigacao = irrig.cod
) TO '{features}tipo_irrigacao.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [50]:
## Feature Engineering
# Tipo Agricultura
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_TIPO_AGRICULTURA as cod_agricultura
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    agri AS (
        SELECT "#CODIGO" AS cod, DESCRICAO AS tipo_agricultura
        FROM read_parquet('{inpath}tipo_agricultura.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, 
        CASE
            WHEN agri.tipo_agricultura IS NULL THEN NULL
            WHEN agri.tipo_agricultura IN ('Não se aplica','Floresta Plantada','Floresta Nativa') THEN 'Não se aplica'
            ELSE agri.tipo_agricultura
        END AS tipo_agricultura
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN agri ON op.cod_agricultura = agri.cod
) TO '{features}tipo_agricultura.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [57]:
## Feature Engineering
# Tipo Ciclo/Fase
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_FASE_CICLO_PRODUCAO as cod_ciclo
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    ciclo AS (
        SELECT "#CODIGO" AS cod, DESCRICAO AS tipo_ciclo
        FROM read_parquet('{inpath}fase_ciclo_producao.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, 
        CASE
            WHEN ciclo.tipo_ciclo IS NULL THEN NULL
            WHEN ciclo.tipo_ciclo IN ('Creche - ENCERRADO','Cria ou Multiplicação','Cria/Creche - ENCERRADO','Retenção de Matrizes','Cria/Recria','Recria') THEN 'Cria/Recria'
            WHEN ciclo.tipo_ciclo IN ('Engorda','Terminação - ENCERRADO','Engorda em confinamento') THEN 'Engorda'
            WHEN ciclo.tipo_ciclo IN ('Cria/Recria/Engorda (Ciclo Completo)','Cria/Recria/Engorda - ENCERRADO','Cria e Engorda','Recria e Engorda','Recria/Terminação - ENCERRADO') THEN 'Ciclo completo'
            WHEN ciclo.tipo_ciclo IN (
                'Corte Raso Final','Corte Raso Intermediário - ENCERRADO','Demais Cortes - ENCERRADO','Primeiro Corte - ENCERRADO','Segundo Corte - ENCERRADO','Terceiro Corte - ENCERRADO','Quarto Corte - ENCERRADO'
            ) THEN 'Corte'
             WHEN ciclo.tipo_ciclo IN ('Semestral','Anual','Bienal') THEN ciclo.tipo_ciclo
            ELSE 'Outros ciclos'
        END AS tipo_ciclo
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN ciclo ON op.cod_ciclo = ciclo.cod
) TO '{features}tipo_ciclo.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [62]:
## Feature Engineering
# Tipo Integração
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_TIPO_INTGR_CONSOR as cod_integracao
        FROM read_parquet('{inpath}operacao_basica.parquet')
    ),
    integ AS (
        SELECT "#CODIGO" AS cod, DESCRICAO AS tipo_integracao
        FROM read_parquet('{inpath}tipo_integracao.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, integ.tipo_integracao
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
    JOIN integ ON op.cod_integracao = integ.cod
) TO '{features}tipo_integracao.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [68]:
## Feature Engineering
# Aliquota Proagro
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, VL_ALIQ_PROAGRO as aliq_proagro
        FROM read_parquet('{inpath}operacao_basica.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, op.aliq_proagro
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
) TO '{features}aliquota_proagro.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [69]:
## Feature Engineering
# Juros
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, VL_JUROS as juros
        FROM read_parquet('{inpath}operacao_basica.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, op.juros
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
) TO '{features}juros.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [70]:
## Feature Engineering
# Área Informada
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    op AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, VL_AREA_INFORMADA as area_informada
        FROM read_parquet('{inpath}operacao_basica.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, op.area_informada
    FROM safra
    JOIN op ON safra.ref_bacen = op.ref_bacen AND safra.nu_ordem = op.nu_ordem
) TO '{features}area_informada.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
## REFAZER

## Feature Engineering
# Município
# Identificador: Ref_Bacen e Nu_Ordem 
# Descrição:

sql = f"""
COPY (
    WITH safra AS (
        SELECT ref_bacen, nu_ordem, target_18m
        FROM read_parquet('{labels}target_18m.parquet')
    ),
    comp AS (
        SELECT "#REF_BACEN" AS ref_bacen, NU_ORDEM AS nu_ordem, CD_IBGE_MUNICIPIO as municipio
        FROM read_parquet('{inpath}complemento_renegociacoes.parquet')
    ),
    muni AS (
        SELECT COD_MUNICIPIO_IBGE AS cod, NOME_MUNICIPIO as municipio
        FROM read_parquet('{inpath}municipios.parquet')
    )
    SELECT safra.ref_bacen, safra.nu_ordem, safra.target_18m, muni.municipio
    FROM safra
    JOIN comp ON safra.ref_bacen = comp.ref_bacen AND safra.nu_ordem = comp.nu_ordem
    JOIN muni ON comp.municipio = muni.cod
) TO '{features}municipio.parquet' (FORMAT PARQUET)
"""
db.execute(sql)

In [ ]:
VL_REC_PROPRIO -- VER % DE RECEITA PROPRIA
RENEG/PRORROG ATIVAS
RENEG/PRORROG LTM
CONTRATOS ATIVOS
CONTRATOS EM ATRASO
SALDO EM ATRASO
%SALDO RENEG/PRORROG
NUMERO DE PARCELAS
SOLICITACOES PROAGRO
ATIVACOES PROAGRO
%AREA PERDIDA
NUMERO PROPRIEDADES
PROPRIEDADE PRÓPRIA/ARRENDADA
%AREA PRÓPRIA/ARRENDADA
SALDO A VENCER

In [99]:
db.execute(f"SELECT * FROM read_parquet('{inpath}sumula_julgamento.parquet') LIMIT 5").df()

,#REF_BACEN,NU_ORDEM,VL_COB_ANT_PARCELA_INVEST_PROAGRO_MAIS,VL_REMU_ENCARR_COMPROV_PERDAS,CD_STATUS,VL_RECEITAS_CONSIDERADAS,VL_COBERTURA_ANT_REC_PROPRIOS,VL_DEMAIS_DESPESAS_COMPROV_PERD,VL_CRED_CUSTEIO_USADO,VL_DEMAIS_DESP_ANT_COMP_PER,...,DT_DECISAO,DT_BASE,VL_COBERTURA_ANT_CREDITO_CUSTEIO,VL_BONUS_PGPAF,VL_DEDUCOES_LEGAIS,VL_ORCAMENTO_ENQUADRADO,NU_DIAS_UTEIS_ATRASO_PERITO,IB_SEGUNDA_VISTORIA,_source_object,_loaded_at
0,509122088,1,0.0,0.00,1,4463.46,0.0,0.0,7232.50,0.0,...,2020-02-28,2020-02-28,0.0,0.0,0.0,7382.00,None,None,sicor/sumula_julgamento/ingest_month=2026-06/S...,2026-06-27T15:02:15.169369
1,509122089,1,0.0,330.00,1,14190.75,0.0,0.0,19930.51,0.0,...,2020-03-11,2020-03-11,0.0,0.0,0.0,24734.71,None,None,sicor/sumula_julgamento/ingest_month=2026-06/S...,2026-06-27T15:02:15.169369
2,509122117,1,0.0,1350.00,1,81396.00,0.0,0.0,209143.63,0.0,...,2020-05-05,2020-05-05,0.0,0.0,0.0,220320.24,None,None,sicor/sumula_julgamento/ingest_month=2026-06/S...,2026-06-27T15:02:15.169369
3,509122131,1,0.0,1350.00,1,96066.00,0.0,0.0,165304.14,0.0,...,2020-04-16,2020-04-16,0.0,0.0,0.0,194487.63,None,None,sicor/sumula_julgamento/ingest_month=2026-06/S...,2026-06-27T15:02:15.169369
4,509122136,1,0.0,372.35,1,11068.74,0.0,0.0,30585.25,0.0,...,2020-04-23,2020-04-23,0.0,0.0,0.0,37235.25,None,None,sicor/sumula_julgamento/ingest_month=2026-06/S...,2026-06-27T15:02:15.169369


In [100]:
db.execute(f"SELECT * FROM read_parquet('{inpath}municipios.parquet')").df()

,#COD_MUNICIPIO_CADMU,NOME_MUNICIPIO,COD_MUNICIPIO_IBGE,COD_UF,NOME_UF,COD_UF_IBGE,_source_object,_loaded_at
0,39480,ROSÁRIO,2109601,MA,MARANHÃO,21,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
1,32889,LAGOA SALGADA,2406601,RN,RIO GRANDE DO NORTE,24,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
2,40338,CRUZÁLIA,3513306,SP,SÃO PAULO,35,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
3,8679,ANAURILÂNDIA,5000807,MS,MATO GROSSO DO SUL,50,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
4,55189,JOCA MARQUES,2205458,PI,PIAUÍ,22,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
...,...,...,...,...,...,...,...,...
5568,51970,IPIGUÁ,3521150,SP,SÃO PAULO,35,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
5569,56542,PACARAIMA,1400456,RR,RORAIMA,14,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
5570,6059,PLANALTO,3539608,SP,SÃO PAULO,35,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
5571,50892,DORMENTES,2605152,PE,PERNAMBUCO,26,sicor/municipios/ingest_month=2026-06/Municipi...,2026-06-30T01:37:45.086444
